## Section 1: Load Final Artifacts

In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb

In [2]:
ranker = joblib.load(
    "../artifacts/final_ranker.pkl"
)

FEATURE_COLUMNS = joblib.load(
    "../artifacts/final_ranker_features.pkl"
)

In [3]:
interaction_df = pd.read_parquet(
    "../data/processed/model_df.parquet"
)

product_features = pd.read_parquet(
    "../artifacts/product_features.parquet"
)

popularity_df = pd.read_parquet(
    "../artifacts/popularity_recommender.parquet"
)

## Section 2: Rebuild Customer Features

In [4]:
customer_spend = (
    interaction_df
    .groupby("customer_unique_id")["price"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "price": "customer_total_spend"
        }
    )
)

In [5]:
customer_frequency = (
    interaction_df
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="customer_purchase_count")
)

In [6]:
customer_category = (
    interaction_df
    .groupby(
        [
            "customer_unique_id",
            "product_category_name"
        ]
    )
    .size()
    .reset_index(name="cnt")
)

In [7]:
customer_favorite_category = (
    customer_category
    .sort_values(
        "cnt",
        ascending=False
    )
    .drop_duplicates(
        subset=["customer_unique_id"]
    )
)

In [8]:
customer_favorite_category = (
    customer_favorite_category[
        [
            "customer_unique_id",
            "product_category_name"
        ]
    ]
    .rename(
        columns={
            "product_category_name":
            "favorite_category"
        }
    )
)

In [9]:
customer_features_eval = (
    customer_spend
    .merge(
        customer_frequency,
        on="customer_unique_id"
    )
    .merge(
        customer_favorite_category,
        on="customer_unique_id"
    )
)

## Section 3: Build Product Table

In [10]:
product_features_eval = (
    product_features
    .merge(
        popularity_df[
            [
                "product_id",
                "popularity_score"
            ]
        ],
        on="product_id",
        how="left"
    )
)

In [11]:
product_features_eval[
    "popularity_score"
] = (
    product_features_eval[
        "popularity_score"
    ].fillna(0)
)

## Section 4: Recommendation Function

In [12]:
def recommend_for_customer(
    customer_id,
    top_k=10
):

    customer_row = customer_features_eval[
        customer_features_eval[
            "customer_unique_id"
        ] == customer_id
    ]

    if customer_row.empty:
        return None

    customer_row = customer_row.iloc[0]

    purchased_products = set(
        interaction_df[
            interaction_df[
                "customer_unique_id"
            ] == customer_id
        ]["product_id"]
    )

    candidates = product_features_eval[
        ~product_features_eval[
            "product_id"
        ].isin(
            purchased_products
        )
    ].copy()

    candidates[
        "customer_total_spend"
    ] = customer_row[
        "customer_total_spend"
    ]

    candidates[
        "customer_purchase_count"
    ] = customer_row[
        "customer_purchase_count"
    ]

    candidates[
        "is_favorite_category_match"
    ] = (
        candidates[
            "product_category_name"
        ]
        ==
        customer_row[
            "favorite_category"
        ]
    ).astype(int)

    X = candidates[
        FEATURE_COLUMNS
    ]

    candidates["score"] = ranker.predict(X)

    return (
        candidates
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
    )

## Section 5: Pick Evaluation Customers

In [13]:
top_customer = (
    customer_spend
    .sort_values(
        "customer_total_spend",
        ascending=False
    )
    .iloc[0]["customer_unique_id"]
)

top_customer

'0a0a92112bd4c708ca5fde585afaa872'

In [14]:
medium_customer = (
    customer_spend[
        customer_spend[
            "customer_total_spend"
        ].between(
            100,
            200
        )
    ]
    .sample(1, random_state=42)
    .iloc[0]["customer_unique_id"]
)

medium_customer

'8ce938b7447a8327dadf2a50c43ac271'

In [15]:
low_customer = (
    customer_spend[
        customer_spend[
            "customer_total_spend"
        ] < 50
    ]
    .sample(1, random_state=42)
    .iloc[0]["customer_unique_id"]
)

low_customer

'daaa4bd4b2ac37cd00efc852878b626a'

## Section 6: Inspect Their Purchase History

In [16]:
interaction_df[
    interaction_df[
        "customer_unique_id"
    ] == top_customer
][
    [
        "product_id",
        "product_category_name",
        "price"
    ]
]

,product_id,product_category_name,price
4243,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4244,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4245,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4246,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4247,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4248,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4249,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4250,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0


## Section 7: Generate Recommendations

In [17]:
recommend_for_customer(
    top_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
32098,ferramentas_jardim,0.683908,1.184673
26862,industria_comercio_e_negocios,0.011494,0.760535
18016,esporte_lazer,0.011494,0.541249
14051,ferramentas_jardim,0.904215,0.311186
8290,informatica_acessorios,0.620690,-0.346707
10963,informatica_acessorios,0.197318,-0.483128
27147,telefonia_fixa,0.026820,-0.721524
15623,telefonia_fixa,0.019157,-0.726683
4598,ferramentas_jardim,0.726054,-0.845454
30892,moveis_decoracao,0.091954,-1.131997


In [18]:
recommend_for_customer(
    medium_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
4855,informatica_acessorios,0.057471,1.534776
31916,informatica_acessorios,0.000000,1.134283
21395,informatica_acessorios,0.333333,0.846658
8290,informatica_acessorios,0.620690,0.831492
22046,informatica_acessorios,0.000000,0.794386
26252,informatica_acessorios,0.172414,0.598058
8580,informatica_acessorios,0.030651,0.548879
32046,informatica_acessorios,0.063218,0.532527
9613,informatica_acessorios,0.000000,0.513803
8662,informatica_acessorios,0.049808,0.491976


In [19]:
recommend_for_customer(
    low_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
28309,informatica_acessorios,0.057471,1.654879
27548,informatica_acessorios,0.114943,1.557999
16192,informatica_acessorios,0.007663,1.280979
17089,informatica_acessorios,0.030651,1.161374
16748,informatica_acessorios,0.063218,1.122240
12853,informatica_acessorios,0.009579,1.096869
13161,informatica_acessorios,0.009579,1.096869
27184,informatica_acessorios,0.013410,1.042864
8213,informatica_acessorios,0.057471,1.042608
13765,informatica_acessorios,0.013410,1.016497


In [20]:
print("TOP CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == top_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

TOP CUSTOMER


,product_category_name,price
4243,telefonia_fixa,1680.0
4244,telefonia_fixa,1680.0
4245,telefonia_fixa,1680.0
4246,telefonia_fixa,1680.0
4247,telefonia_fixa,1680.0
4248,telefonia_fixa,1680.0
4249,telefonia_fixa,1680.0
4250,telefonia_fixa,1680.0


In [21]:
print("MEDIUM CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == medium_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

MEDIUM CUSTOMER


,product_category_name,price
31604,informatica_acessorios,105.0


In [22]:
print("LOW CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == low_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

LOW CUSTOMER


,product_category_name,price
3536,informatica_acessorios,43.89
